**Jacob Petty, Sardor Nodirov, and Saad Khan**

Spring 2026

CS 443: Bio-inspired Machine Learning

Project 2: Predictive Coding

#### Week 2: Predictive coding: classification and image generation

With the Dense PCN built, let's analyze its classification accuracy on MNIST and CIFAR-10 and explore its generative image capabilities. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.show()
plt.rcParams.update({'font.size': 18})

np.set_printoptions(suppress=True, precision=3)

%load_ext autoreload
%autoreload 2

## Task 3: Dense PCN and MNIST

Now you are ready to train your Dense PCN on MNIST!

In addition to optimizing the classification accuracy, you will:
1. visualize the "ideal" digit that each output neuron of a trained net "expects to see".
2. leverage this generative capability of PCNs to "fill in" blanked out parts of test samples.

In [ ]:
from image_datasets import get_dataset, train_val_split
from dense_pcn import DensePCN

### 3a. Train and evaluate classification accuracy on MNIST

In the cell below, train a Dense PCN with 1 hidden layer on *centered* preprocessed MNIST samples with the default train/val/test split.

Your goal is to play with the following hyperparameters to obtain **≥75% accuracy on the test set**:
- number of epochs
- batch size
- number of units

Fix your learning rate to `1e-4` and use the default number of train/test feedback iterations.

Be sure to print out your test accuracy.


**Notes:**
- If your training is working properly, you should see your training loss consistently decline.
- You should not need to train for many epochs to hit the accuracy target. 
- I suggest setting a random seed so that similar to the test code your results do not fluctuate based on differences in initial random weights between runs.

### 3b. Hyperparameter tuning: test steps

With the same hyperparameters that you used in the previous subtask, run inference on your Dense PCN to compute the test accuracy with the following number of test steps: `1, 10, 20, 30, ..., 100`.

Make a plot showing the MNIST test accuracy in each case.

### 3c. Questions

**Question 2:** (a) What number of test steps produced the optimal test accuracy for you? (b) What was the best test accuracy that you achieved and how does that compare to the baseline that you found in the previous subtask?

**Question 3:** Compared to the baseline number of steps, why would the larger (or smaller) number of steps lead to improved accuracy in the PCN?

**Answer 2:** YOUR ANSWER HERE

**Answer 3:** YOUR ANSWER HERE

### 3d. Expected class images

Follow the inline instructions to finish the implementation of the `dream_input` method in the `DensePCN` class. Afterwards, run the code provided below to visualize the expected input for each output neuron of your PCN!

*You should see a very cool ~30 sec animation when running the cell below.*

In [ ]:
# Keep me
class_names = tf.range(10)
net.dream_input(class_names)

### 3e. Questions

**Question 4:** Describe what you see in the animation and interpret what it means (i.e. why do you see what you see in each figure panel in the order in which you see it).

**Answer 4:** YOUR ANSWER HERE

### 3f. Filling in occluded parts of images

This subtask focuses on a neat application of the generative capabilities of PCNs — filling in or completing parts of test images that are *missing* (i.e. blanked out). 

#### (i) Visualizing occluded MNIST images

Run the cell below to generate versions of test set images in which the top half of pixels is blanked (**masked**) out.

*For the first few MNIST test set images, you should see the original image (top), occluded image (middle) and the occlusion mask used to generate the occluded image (bottom). In this plotting convention, parts of the mask that zero out corresponding parts of each image sample are black (0s) and parts that preserve the image data are white (1s).*

In [ ]:
from image_datasets import occlude_images

In [ ]:
x_test_occ, masks = occlude_images(x_test, region='top')

items = [x_test, x_test_occ, masks]
labels = ['Raw', 'Occluded', 'Mask']

num_samples = 5
fig, axes = plt.subplots(3, num_samples)
for i in range(num_samples):
    for j in range(3):
        axes[j, i].imshow(tf.reshape(items[j][i], (28, 28)), cmap='gray')
        axes[j, i].set_title(labels[j], fontsize=12)
        axes[j, i].grid(False)
        axes[j, i].set_xticks([])
        axes[j, i].set_yticks([])

plt.show()

#### (ii) Add support for dreaming in masked parts of images

We will use your PCN's generative dreaming capability to "fill in" the occluded parts of the image based on the available information and the net's expectation for what should appear in the occluded image regions. For this to work, we need to slightly modify how your `InputPCNLayer` modifies its state: we ONLY want to have the net dream in MASKED parts of the image (non-occluded raw pixels should remain untouched as the state is updated).

In `InputPCNLayer` make the following small updates:
1. Implement `set_mask` to create an instance variable to store the masks for each test set mini-batch.
2. Modify `update_state` to use the occlusion mask to zero out the *top-down prediction error* in places where the input images were NOT MASKED by the occlusion mask (i.e. in places where there is actual useful image data). If there is no mask, modify the state like usual.

#### (iii) Finish method to fill in masked regions of image and animate the process

Follow the inline instructions to finish the implementation of the `complete_input` method in the `DensePCN` class, which will fill in the occluded parts of MNIST images.

Run the following code to animate the filling in process. *You should see a ~30 sec animation.*

**Note:** Replace `net` with your trained net's name.


In [ ]:
min_ind = 0
max_ind = 10
net.complete_input(x_test_occ[min_ind:max_ind],
                   masks[min_ind:max_ind],
                   y_test[min_ind:max_ind])

### 3g. Questions

**Question 5:**

(a) How closely do the filled in images resemble the original images?

(b) How sensible/plausible are the filled in content by the network?

**Answer 5:** YOUR ANSWER HERE

## Task 4: Dense PCN and CIFAR-10

This task focuses on evaluating the classification accuracy of your dense PCN on CIFAR-10. You will also visualize the expected images corresponding to each output neuron.

### 4a. Train and evaluate classification accuracy on CIFAR-10

In the cell below, train a Dense PCN with 1 hidden layer on *centered* preprocessed CIFAR-10 samples with the default train/val/test split.

Use default hyperparameters, except for the following:
- `10` epochs.
- Batch size of `256`.
- Learning rate of `1e-4`.

Print out the accuracy that you obtain on the test set. If everything is working as expected, you should be able to obtain **test accuracy ≥ 25%**.

### 4b. Visualize expected inputs for each output neuron

Running the following code should show an animation of the expected input according to each output neuron of your trained PCN.

In [ ]:
cifar_class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
net.dream_input(cifar_class_names, image_dims=(32, 32, 3))

### 4c. Questions

**Question 6:** Why are the expected inputs for each class blurrier than those that you generated for MNIST?

**Question 7:** Why might the blurriness of expected images of each class hinder the potential for feedback in the PCN to raise the classification accuracy?

**Answer 6:** YOUR ANSWER HERE

**Answer 7:** YOUR ANSWER HERE